# Fine tuning the sensitivity classifier

### importing libraries

> **⚠️ Warning:**  
> The code that makes use of `DistilBert` based APIs from the `transformers` library by Hugging Face, was derived from the documentation for the API which can be found here:  
> https://huggingface.co/transformers/v4.4.2/model_doc/distilbert.html

In [ ]:
pip install tensorflow==2.16.1 tf-keras==2.16.0 transformers==4.40.0 accelerate datasets pandas scikit-learn

In [5]:
import transformers
print(transformers.__version__)

4.40.0


In [7]:
import tensorflow as tf
print(tf.__version__)

2.16.1


In [ ]:
import pandas as pd
import json
import tensorflow as tf
from transformers import DistilBertTokenizerFast
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import TFDistilBertForSequenceClassification


### Creating combined dataset

In [5]:
# Load Fn_dataset.jsonl
fn_df = pd.read_json('Fn_dataset.jsonl', lines=True)
fn_df['text'] = fn_df['prompt']
fn_df['label'] = fn_df['label'].str.lower().map({'sensitive': 'encrypt', 'benign': 'not_encrypt'})

# Load sensitive_data_password.csv
sensitive_df = pd.read_csv('sensitive_data_password.csv')
sensitive_df['text'] = sensitive_df['text']
sensitive_df['label'] = sensitive_df['label'].map({True: 'encrypt', False: 'not_encrypt', 'True': 'encrypt', 'False': 'not_encrypt'})

# Load synthetic_dataset.csv
synthetic_df = pd.read_csv('synthetic_dataset.csv')
synthetic_df['text'] = synthetic_df['prompt_clean']
synthetic_df['label'] = synthetic_df['label'].map({'harmful': 'encrypt', 'safe': 'not_encrypt'})

# Combine all and drop empty rows
all_data = pd.concat([fn_df[['text', 'label']], sensitive_df[['text', 'label']], synthetic_df[['text', 'label']]], ignore_index=True)
all_data = all_data.dropna()

# Save combined dataset
all_data.to_csv('combined_dataset.csv', index=False)

#### Loading and cleaning the combined dataset

In [6]:
# Load the combined dataset
combined_df = pd.read_csv('combined_dataset.csv')

# Replace "Example" and everything after it with "."
# The synthetic data included the word "Example" 4500 times so it has to be removed
combined_df['text'] = combined_df['text'].str.replace(r'Example.*', '', regex=True)

# Save the modified dataset
combined_df.to_csv('combined_dataset.csv', index=False)

### Preparing the dataset

#### Chosing the context size

In [10]:
import pandas as pd
df = pd.read_csv('combined_dataset.csv')
# Count tokens
# Approximate token lengths based on white space.
df['token_count'] = df['text'].apply(lambda x: len(x.split())) 

print(df['token_count'].describe())
print(f"95th percentile: {df['token_count'].quantile(0.95)}")

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
count    6038.000000
mean        8.355747
std         9.224994
min         1.000000
25%         5.000000
50%         7.000000
75%         8.000000
max       359.000000
Name: token_count, dtype: float64
95th percentile: 25.0


There are a total of 6038 samples in the combined dataset, of which 95% tokenize into 25 tokens (estimate based on tokenizng by white space). So going with a standard 128 tokens should be fine.

#### Encoding the labels into integers

In [9]:
# Load the combined dataset
combined_df = pd.read_csv('combined_dataset.csv')

# Models does better with numerical classes
# Encode labels to integers
label_encoder = LabelEncoder()
combined_df['label_id'] = label_encoder.fit_transform(combined_df['label'])
# combined_df.to_csv('combined_dataset_with_label_id.csv', index=False) 
label2id = {label: int(idx) for idx, label in enumerate(label_encoder.classes_)}
id2label = {int(idx): label for idx, label in enumerate(label_encoder.classes_)}

# Train - validation split
# No point in a test set as I can't really change any model parameters to optimize based on performance
train_df, val_df = train_test_split(combined_df, test_size=0.1, random_state=42, stratify=combined_df['label_id'])

# Show label mapping and dataset sizes
print('Label mapping:', label2id)
print('Train size:', len(train_df), 'Val size:', len(val_df))

Label mapping: {'encrypt': 0, 'not_encrypt': 1}
Train size: 5434 Val size: 604


#### tokenizing the dataset

In [12]:
# Using a lightweight BERT variant for mobile called DistilBERT 
# This model does not distinguish case e.g. interprets "english" and "English" as one on the same
MODEL_NAME = 'distilbert-base-uncased'
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

# Using the maximum length determined earlier
# Kept small for TFLite
MAX_LEN = 128

def tokenize_texts(texts):
    return tokenizer(list(texts), padding='max_length', truncation=True, max_length=MAX_LEN, return_tensors='tf')

# tokenize training and validation sets
train_encodings = tokenize_texts(train_df['text'])
val_encodings = tokenize_texts(val_df['text'])

# Convert labels to tensors
# this is done to make the labels compatible with TensorFlow training
# as model.fit() works best with TensorFlow tensors 
train_labels = tf.convert_to_tensor(train_df['label_id'].values)
val_labels = tf.convert_to_tensor(val_df['label_id'].values)

In [13]:
# Dataset is small enough (6038 samples) for TensorFlow to train on CPU 
# Use this to check if GPUs are available anyway
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU'))) 
print(tf.config.list_physical_devices('GPU'))

Num GPUs Available:  0
[]


### Training the model

> **⚠️ Warning:**  
> Running the model training on CPU will take 30 - 45 minutes to complete. <br>
> Running the model training with a GPU will take 3 - 5 minutes to complete.

In [15]:
# Load pretrained DistilBERT for sequence classification
model = TFDistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(label2id), id2label=id2label, label2id=label2id)

# Prepare tf.data.Dataset objects for efficient training
BATCH_SIZE = 16

# Covert to TensorFlow dataset for GPU acceleration
train_dataset = tf.data.Dataset.from_tensor_slices((dict(train_encodings), train_labels)).shuffle(1000).batch(BATCH_SIZE)
val_dataset = tf.data.Dataset.from_tensor_slices((dict(val_encodings), val_labels)).batch(BATCH_SIZE)

# Compile model using Adam optimizer and crossentrophy as a loss function
optimizer = tf.keras.optimizers.Adam(learning_rate=3e-5)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metrics = [tf.keras.metrics.SparseCategoricalAccuracy()]
model.compile(optimizer=optimizer, loss=loss, metrics=metrics)

# Train model
EPOCHS = 3
history = model.fit(train_dataset, validation_data=val_dataset, epochs=EPOCHS)

# Save the trained model in SavedModel format for TFLite conversion
model.save_pretrained('distilbert_finetuned_tf', saved_model=True)

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForSequenceClassification: ['vocab_transform.bias', 'vocab_layer_norm.bias', 'vocab_layer_norm.weight', 'vocab_transform.weight', 'vocab_projector.bias']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFDistilBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['pre_classifier.weight', 'pre_classifier.bias', 'classifier.weight', 'classifier.bias']
You should 

Epoch 1/3
340/340 [==============================] - 805s 2s/step - loss: 0.0944 - sparse_categorical_accuracy: 0.9639 - val_loss: 0.0500 - val_sparse_categorical_accuracy: 0.9801
Epoch 2/3
340/340 [==============================] - 781s 2s/step - loss: 0.0159 - sparse_categorical_accuracy: 0.9952 - val_loss: 0.0106 - val_sparse_categorical_accuracy: 0.9967
Epoch 3/3
340/340 [==============================] - 755s 2s/step - loss: 0.0078 - sparse_categorical_accuracy: 0.9969 - val_loss: 0.0173 - val_sparse_categorical_accuracy: 0.9917


INFO:tensorflow:Assets written to: distilbert_finetuned_tf/saved_model/1/assets


INFO:tensorflow:Assets written to: distilbert_finetuned_tf/saved_model/1/assets


#### Training performance summary

Epoch 1/3 <br>
Training dataset: <br>
loss: 0.0944 - sparse_categorical_accuracy: 0.9639 <br>
Validation dataset: <br>
val_loss: 0.0500 - val_sparse_categorical_accuracy: 0.9801

Epoch 2/3 <br>
Training dataset: <br>
loss: 0.0159 - sparse_categorical_accuracy: 0.9952 <br>
Validation dataset: <br>
val_loss: 0.0106 - val_sparse_categorical_accuracy: 0.9967

Epoch 3/3 <br>
Training dataset: <br>
loss: 0.0078 - sparse_categorical_accuracy: 0.9969 <br>
Validation dataset: <br>
val_loss: 0.0173 - val_sparse_categorical_accuracy: 0.9917

### Converting fine tuned model to TensorFlow Lite

In [16]:
# Specifying storage locations
saved_model_dir = 'distilbert_finetuned_tf/saved_model/1'
tflite_model_path = 'distilbert_finetuned.tflite'

# Convert to TFLite with dynamic range quantization (for mobile)
converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open(tflite_model_path, 'wb') as f:
    f.write(tflite_model)

print(f'TFLite model saved to {tflite_model_path}')

W0000 00:00:1777501849.383273     464 tf_tfl_flatbuffer_helpers.cc:390] Ignored output_format.
W0000 00:00:1777501849.386247     464 tf_tfl_flatbuffer_helpers.cc:393] Ignored drop_control_dependency.
2026-04-29 23:30:49.395119: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: distilbert_finetuned_tf/saved_model/1
2026-04-29 23:30:49.596068: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-04-29 23:30:49.596180: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: distilbert_finetuned_tf/saved_model/1
2026-04-29 23:30:49.316955: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:388] MLIR V1 optimization pass is not enabled
2026-04-29 23:30:49.335359: I tensorflow/cc/saved_model/loader.cc:234] Restoring SavedModel bundle.
2026-04-29 23:30:49.871546: I tensorflow/cc/saved_model/loader.cc:218] Running initialization op on SavedModel bundle at path: distilbert_finetuned_tf/saved_model/1
20

TFLite model saved to distilbert_finetuned.tflite


#### Saving tokenizer and label mapping for Scrambler integration

In [17]:
# Save label mapping
with open('label2id.json', 'w') as f:
    json.dump(label2id, f)

with open('id2label.json', 'w') as f:
    json.dump(id2label, f)

# Save tokenizer configuration and vocabulary
tokenizer.save_pretrained('distilbert_tokenizer', legacy_format=False)

print('Tokenizer and label mapping saved.')

Tokenizer and label mapping saved.
